# 09 · Removendo o viés: Neutralize, Equalize e Soft debias (Bolukbasi et al., 2016, §6–§8)

> **Disciplina:** Representações de Linguagem — *como transformar palavra em vetor, o que esse vetor carrega e o viés que vem junto* (BLIS 2S26).
> Este notebook faz parte de uma sequência (`00` → `13`). Código de apoio em `src/replang`; interface interativa em `app/` (`uv run replang app`).


## Objetivos do *debias* (§1)
1. **Reduzir o viés**: (a) palavras neutras equidistantes dos pares de gênero (*nurse* igualmente perto de *he* e *she*); (b) reduzir associações de gênero entre palavras neutras (viés indireto).
2. **Manter a utilidade**: (a) preservar associações não ligadas a gênero (moda, futebol); (b) preservar o gênero definicional (*man–father*, *she:grandmother :: he:grandfather*).

## Os algoritmos
* **Passo 1 — Identificar o subespaço de viés** $B$ (notebook 08): $k$ primeiras componentes da PCA das diferenças centradas dos pares definicionais ($k = 1$: a direção $g$).
* **Passo 2a — Hard de-biasing** = *Neutralize* + *Equalize*.
* **Passo 2b — Soft bias correction**: transformação linear $T$ que preserva produtos internos enquanto reduz a projeção das neutras em $B$, com compromisso $\lambda$.

Implementações: `replang.bias.debias.hard_debias`, `soft_debias`, `pair_bias`.

In [1]:
# Configuração comum a todos os notebooks ---------------------------------------------
import sys, warnings, os
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import plotly.io as pio
pio.renderers.default = "plotly_mimetype+notebook_connected"  # JupyterLab renderiza offline; HTML carrega plotly.js via CDN (notebooks leves)
pd.set_option("display.max_colwidth", 80); pd.set_option("display.width", 160)
from replang import PATHS, WordVectors, settings
FAST = settings.fast  # REPLANG_FAST=1 reduz épocas/tamanhos
print("raiz:", ROOT, "| fast:", FAST)

raiz: /home/user/linguagem-blis2s26 | fast: False


In [2]:
from replang.data.embeddings import load_glove_en
from replang.data.lexicons import EN
from replang.bias import gender_direction, direct_bias, indirect_bias, pair_bias, hard_debias, soft_debias, project, extremes
lex = EN.get()
en = load_glove_en().subset([w for w in load_glove_en().words if w.isalpha() and len(w) < 20], "GloVe 50d").normalized()
g = gender_direction(en, lex.definitional_pairs)
B = g[None, :]
profs = [p for p in lex.professions if p in en]
neutral = [w for w in en.words if w not in lex.all_gender_words()]
print(en, "| direção g pronta | profissões:", len(profs))

WordVectors(name='GloVe 50d', n=45772, dim=50) | direção g pronta | profissões: 285


## 1. Neutralize (§6, passo 2a)

Para cada palavra neutra $w \in N$: remover a projeção no subespaço e renormalizar,

$$\vec w := \frac{\vec w - \vec w_B}{\|\vec w - \vec w_B\|}, \qquad \vec w_B = \sum_{j=1}^{k} (\vec w\cdot b_j)\, b_j .$$

É uma **projeção ortogonal**: tudo o que $w$ "sabia" fora de $B$ é mantido; a coordenada de gênero vira zero. O conjunto $N$ é definido como complemento das palavras específicas de gênero ($N = W \setminus S$, §7).

In [3]:
def neutralize(v, B):
    v_B = (v @ B.T) @ B
    out = v - v_B
    return out / np.linalg.norm(out)
nurse = en["nurse"]
print(f"cos(nurse, g) antes = {float(nurse @ g):+.3f}  →  depois = {float(neutralize(nurse, B) @ g):+.3f}")
print("vizinhos de nurse antes :", [w for w, _ in en.most_similar("nurse", topn=8)])

cos(nurse, g) antes = +0.262  →  depois = -0.000
vizinhos de nurse antes : ['doctor', 'nurses', 'dentist', 'pregnant', 'pediatrician', 'therapist', 'surgeon', 'nursing']


## 2. Equalize (§6, passo 2a)

Para cada conjunto de igualdade $E$ (ex.: {grandmother, grandfather}, {guy, gal}): as palavras ficam **simétricas** em torno de um centro $\nu$ fora de $B$, mantendo norma 1:

$$\mu := \frac{1}{|E|}\sum_{w\in E}\vec w,\qquad \nu := \mu - \mu_B,\qquad \vec w := \nu + \sqrt{1 - \|\nu\|^2}\;\frac{\vec w_B - \mu_B}{\|\vec w_B - \mu_B\|}.$$

Fora de $B$, as duas palavras ficam **iguais** ($\nu$); dentro de $B$, ficam **centradas** (opostas) e reescaladas para que $\|\vec w\| = 1$. Por que centrar? Porque no embedding original *female* tem projeção de gênero maior que *male* (ambas positivas!) — sem centrar, *male* e *female* ficariam idênticas e perderíamos analogias como *father:male :: mother:female*.

**Observação 1 do artigo**: após Neutralize + Equalize, para toda palavra neutra $w$ e todo par $(e_1, e_2)$ de um conjunto de igualdade, $\vec w\cdot\vec e_1 = \vec w\cdot\vec e_2$ e $\|\vec w - \vec e_1\| = \|\vec w - \vec e_2\|$; logo $\mathrm{PairBias} = 0$.

In [4]:
import inspect
from replang.bias import debias
src = inspect.getsource(debias.hard_debias)
print(src[src.index("# Neutralize"):src.index("out.invalidate()")])

# Neutralize: remove a componente em B e renormaliza
    if len(idx):
        sub = V[idx]
        sub = sub - _proj_B(sub, B)
        sub /= np.linalg.norm(sub, axis=1, keepdims=True) + 1e-9
        V[idx] = sub
    # Equalize: cada conjunto E (aqui pares) fica simétrico em torno de ν fora de B
    for pair in equalize_pairs:
        ws = [w for w in pair if w in out.index]
        if len(ws) < 2:
            continue
        ids = [out.index[w] for w in ws]
        mu = V[ids].mean(0)
        nu = mu - _proj_B(mu, B)
        for i in ids:
            w_b = _proj_B(V[i], B)
            mu_b = _proj_B(mu, B)
            diff = w_b - mu_b
            n = np.linalg.norm(diff)
            scale = np.sqrt(max(1 - float(np.dot(nu, nu)), 0.0))
            V[i] = nu + (scale * diff / n if n > 1e-9 else 0.0)
    


In [5]:
hard = hard_debias(en, B, gender_specific=lex.gender_specific, equalize_pairs=lex.equalize_pairs, name="GloVe 50d hard-debiased")
# Verificação da Observação 1
checks = []
for w in ["nurse", "engineer", "receptionist", "homemaker", "programmer"]:
    for a, b in [("she", "he"), ("woman", "man"), ("grandmother", "grandfather")]:
        checks.append({"w": w, "par": f"{a}/{b}", "antes |cos(w,a)−cos(w,b)|": abs(en.similarity(w, a) - en.similarity(w, b)), "depois": abs(hard.similarity(w, a) - hard.similarity(w, b))})
pd.DataFrame(checks).style.format({"antes |cos(w,a)−cos(w,b)|": "{:.3f}", "depois": "{:.1e}"})

,w,par,"antes |cos(w,a)−cos(w,b)|",depois
0,nurse,she/he,0.165,1.2e-07
1,nurse,woman/man,0.144,6.0e-08
2,nurse,grandmother/grandfather,0.191,0.0e+00
3,engineer,she/he,0.074,3.0e-08
4,engineer,woman/man,0.064,3.0e-08
5,engineer,grandmother/grandfather,0.258,0.0e+00
6,receptionist,she/he,0.169,3.0e-08
7,receptionist,woman/man,0.136,0.0e+00
8,receptionist,grandmother/grandfather,0.266,0.0e+00
9,homemaker,she/he,0.169,0.0e+00


In [6]:
print(f"DirectBias_1 (profissões): antes {direct_bias(en, profs, g):.4f} → depois {direct_bias(hard, profs, g):.5f}")
print(f"PairBias (200 neutras × pares definicionais): antes {pair_bias(en, neutral[:200], lex.definitional_pairs):.4f} → depois {pair_bias(hard, neutral[:200], lex.definitional_pairs):.1e}")
print(f"gênero definicional preservado: cos(she,g)={float(hard['she'] @ g):+.2f} cos(he,g)={float(hard['he'] @ g):+.2f} | cos(king,queen)={hard.similarity('king','queen'):.2f} (antes {en.similarity('king','queen'):.2f})")

DirectBias_1 (profissões): antes 0.1785 → depois 0.01488


PairBias (200 neutras × pares definicionais): antes 0.1089 → depois 3.5e-02
gênero definicional preservado: cos(she,g)=+0.27 cos(he,g)=-0.27 | cos(king,queen)=0.76 (antes 0.78)


### 2.1 Geometria do Equalize em 2D

Visualizamos *grandmother*/*grandfather* e *gal*/*guy* no plano formado por $g$ e uma direção ortogonal: antes (assimétricos) e depois (simétricos em torno de $\nu$).

In [7]:
import plotly.graph_objects as go
def plane_coords(wv, words, g):
    X = wv.matrix(words)
    x = X @ g
    resid = X - np.outer(x, g)
    u = resid.mean(0); u -= (u @ g) * g; u /= np.linalg.norm(u)
    return x, resid @ u
fig = go.Figure()
for name, wv, sym in [("antes", en, "circle"), ("depois (hard)", hard, "diamond")]:
    ws = [w for w in ["grandmother", "grandfather", "gal", "guy", "babysitter", "wisdom", "nurse"] if w in wv]
    x, y = plane_coords(wv, ws, g)
    fig.add_scatter(x=x, y=y, mode="markers+text", text=ws, name=name, textposition="top center", marker=dict(size=11, symbol=sym))
fig.add_vline(x=0, line_dash="dash", line_color="gray")
fig.update_layout(template="plotly_white", title="Equalize: pares ficam simétricos em g; neutras (babysit, wisdom) vão para g = 0", xaxis_title="componente em g (gênero)", yaxis_title="componente ortogonal (média)")
fig.show()

## 3. Soft bias correction (§6, passo 2b; Apêndice B)

Em vez de projetar, aprende-se uma transformação linear $T \in \mathbb{R}^{d\times d}$:

$$\min_T\ \|(TW)^\top(TW) - W^\top W\|_F^2 + \lambda\,\|(TN)^\top(TB)\|_F^2$$

O primeiro termo preserva **todos** os produtos internos (a "utilidade"); o segundo empurra a projeção das neutras $N$ no subespaço $B$ para zero. $\lambda \to \infty$ recupera o Neutralize; $\lambda = 0{,}2$ no artigo. O artigo resolve como um **programa semidefinido** (com $X = T^\top T$, e a SVD de $W$ para reduzir a $d\times d$). Nossa versão didática otimiza $T$ por gradiente — mesma função objetivo, solução aproximada.

In [8]:
import time
rows, hists = [], {}
for lam in [0.05, 0.2, 1.0, 5.0]:
    t = time.time()
    soft, T, hist = soft_debias(en, B, neutral_words=profs, lam=lam, steps=200, lr=0.05, sample=2500, seed=0)
    hists[lam] = hist
    rows.append({"λ": lam, "DirectBias (profissões)": direct_bias(soft, profs, g), "cos(king,queen)": soft.similarity("king", "queen"), "cos(nurse,doctor)": soft.similarity("nurse", "doctor"),
                 "desvio médio dos produtos internos": hist[-1]["preservação"] ** 0.5, "s": round(time.time() - t)})
soft_df = pd.DataFrame(rows)
soft_df.style.format({"DirectBias (profissões)": "{:.4f}", "cos(king,queen)": "{:.3f}", "cos(nurse,doctor)": "{:.3f}", "desvio médio dos produtos internos": "{:.4f}"})

,λ,DirectBias (profissões),"cos(king,queen)","cos(nurse,doctor)",desvio médio dos produtos internos,s
0,0.050000,0.0600,0.848,0.826,0.0175,11
1,0.200000,0.0279,0.867,0.831,0.0201,10
2,1.000000,0.0200,0.879,0.828,0.0227,10
3,5.000000,0.0173,0.879,0.807,0.0247,10


In [9]:
import plotly.express as px
h = pd.concat([pd.DataFrame(v).assign(λ=str(k)) for k, v in hists.items()])
fig = px.line(h, x="step", y="viés", color="λ", log_y=True, template="plotly_white", title="Soft debias: termo de viés ao longo da otimização, para vários λ")
fig.show()

## 4. A utilidade é preservada? (§8, Tabela 1 e Fig. 8)

O artigo mede, antes/depois: RG-65 e WS-353 (similaridade) e MSR-analogy — praticamente inalterados (62,3/54,5/57,0 → 62,4/54,1/57,0 hard). E, com a *crowd*: analogias estereotipadas caem de 19 % para 6 % (hard), com o número de analogias apropriadas mantido (Fig. 8). Reproduzimos com o conjunto `questions-words` (restrito, para ser rápido) e com os vizinhos.

In [10]:
from replang.data.analogies import load_analogies
from replang.eval.analogies import evaluate_analogies
qs = load_analogies("en")
rows = []
for name, wv in [("original", en), ("hard-debiased", hard), ("soft λ=0.2", soft_debias(en, B, neutral_words=profs, lam=0.2, steps=200, lr=0.05, sample=2500)[0])]:
    r = evaluate_analogies(wv, qs, restrict=30000, max_per_category=300 if FAST else 600).set_index("categoria")
    rows.append({"embedding": name, "semântica": r.loc["TOTAL semântica", "acurácia"], "sintática": r.loc["TOTAL sintática", "acurácia"], "total": r.loc["TOTAL", "acurácia"], "family": r.loc["family", "acurácia"]})
pd.DataFrame(rows).style.format({"semântica": "{:.1%}", "sintática": "{:.1%}", "total": "{:.1%}", "family": "{:.1%}"})

,embedding,semântica,sintática,total,family
0,original,53.8%,44.0%,47.0%,75.8%
1,hard-debiased,55.2%,43.5%,47.2%,86.6%
2,soft λ=0.2,53.8%,43.7%,46.8%,76.8%


In [11]:
for w in ["nurse", "programmer", "receptionist", "football", "wisdom"]:
    print(f"{w:<13} antes : {[x for x, _ in en.most_similar(w, topn=7)]}")
    print(f"{'':<13} depois: {[x for x, _ in hard.most_similar(w, topn=7)]}")

nurse         antes : ['doctor', 'nurses', 'dentist', 'pregnant', 'pediatrician', 'therapist', 'surgeon']
              depois: ['doctor', 'physician', 'surgeon', 'dentist', 'nurses', 'teacher', 'psychiatrist']
programmer    antes : ['programmers', 'software', 'computer', 'prodigy', 'animator', 'entrepreneur', 'technician']
              depois: ['prodigy', 'programmers', 'software', 'computer', 'technician', 'animator', 'entrepreneur']
receptionist  antes : ['hairdresser', 'waitress', 'janitor', 'housekeeper', 'waiter', 'concierge', 'cashier']
              depois: ['janitor', 'hairdresser', 'waiter', 'cashier', 'concierge', 'housekeeper', 'doorman']
football      antes : ['soccer', 'league', 'basketball', 'club', 'hockey', 'rugby', 'team']
              depois: ['soccer', 'basketball', 'league', 'hockey', 'club', 'rugby', 'team']
wisdom        antes : ['belief', 'sense', 'moral', 'essence', 'faith', 'humility', 'compassion']
              depois: ['belief', 'sense', 'essence', 'moral

In [12]:
# As analogias do artigo: he:doctor :: she:? e she:ovarian_cancer :: he:?
for a, b, c in [("he", "doctor", "she"), ("man", "programmer", "woman"), ("father", "doctor", "mother"), ("she", "nurse", "he")]:
    print(f"{a}:{b} :: {c}:?   antes → {[x for x, _ in en.analogy(a, b, c, topn=4, exclude_words=lex.gender_specific)]}   depois → {[x for x, _ in hard.analogy(a, b, c, topn=4, exclude_words=lex.gender_specific)]}")
print("preservadas (gênero definicional): she:queen :: he:?", [x for x, _ in hard.analogy("she", "queen", "he", topn=3)], "| she:sister :: he:?", [x for x, _ in hard.analogy("she", "sister", "he", topn=3)])

he:doctor :: she:?   antes → ['nurse', 'child', 'pregnant', 'patient']   depois → ['nurse', 'physician', 'child', 'patient']
man:programmer :: woman:?   antes → ['prodigy', 'therapist', 'psychotherapist', 'technician']   depois → ['prodigy', 'programmers', 'software', 'computer']
father:doctor :: mother:?   antes → ['nurse', 'patient', 'pregnant', 'child']   depois → ['nurse', 'physician', 'child', 'patient']
she:nurse :: he:?   antes → ['physician', 'doctor', 'surgeon', 'nurses']   depois → ['doctor', 'physician', 'surgeon', 'dentist']
preservadas (gênero definicional): she:queen :: he:? ['king', 'prince', 'son'] | she:sister :: he:? ['brother', 'son', 'father']


## 5. Viés indireto após o hard debias (§8)

O artigo repete o experimento *softball–football*: *receptionist*, *waitress*, *homemaker* saem do topo; *pitcher* e *footballer* continuam (similaridade legítima).

In [13]:
axis_before = en["softball"] - en["football"]; axis_after = hard["softball"] - hard["football"]
print("softball antes :", extremes(en, axis_before, profs, 6)[0]); print("softball depois:", extremes(hard, axis_after, profs, 6)[0])
print("football antes :", extremes(en, axis_before, profs, 6)[1]); print("football depois:", extremes(hard, axis_after, profs, 6)[1])

softball antes : ['welder', 'lifeguard', 'swimmer', 'ballerina', 'cellist', 'foreman']
softball depois: ['foreman', 'welder', 'promoter', 'cellist', 'lifeguard', 'laborer']
football antes : ['bishop', 'archbishop', 'saint', 'captain', 'boss', 'midfielder']
football depois: ['saint', 'bishop', 'archbishop', 'priest', 'nanny', 'doctor']


## 6. Geração de analogias antes × depois (Fig. 8 do artigo, sem a *crowd*)

Sem avaliadores humanos, contamos quantas das analogias geradas envolvem **profissões** (um proxy grosseiro de "estereótipo ocupacional").

In [14]:
from replang.bias import generate_analogies
prof_set = set(profs)
for name, wv in [("antes", en), ("depois", hard)]:
    an = generate_analogies(wv, "she", "he", delta=1.0, topn=100, restrict=25000, exclude=lex.all_gender_words())
    n_prof = int((an.x.isin(prof_set) | an.y.isin(prof_set)).sum())
    print(f"{name}: {len(an)} analogias geradas; {n_prof} envolvem profissões; exemplos: {list(zip(an.x[:8], an.y[:8]))}")

antes: 100 analogias geradas; 1 envolvem profissões; exemplos: [('amy', 'barry'), ('hannah', 'wallace'), ('julie', 'mike'), ('soprano', 'tenor'), ('alice', 'arthur'), ('caroline', 'thomas'), ('emily', 'james'), ('laura', 'robertson')]


depois: 100 analogias geradas; 3 envolvem profissões; exemplos: [('tuesday', 'thursday'), ('ravens', 'cowboys'), ('april', 'september'), ('february', 'october'), ('seventh', 'sixth'), ('however', 'although'), ('four', 'five'), ('particularly', 'especially')]


## 7. Limitações e a discussão posterior

* **Gonen & Goldberg (2019), "Lipstick on a Pig"**: o hard debias zera a projeção em $g$, mas as palavras "femininas" continuam **agrupadas entre si** (a estrutura relativa sobrevive); um classificador recupera o gênero das neutras com alta acurácia. O viés foi *escondido*, não removido.
* A escolha de $N$, $S$ e dos pares é **subjetiva** e específica da aplicação (§7: *the choice of words is subjective*); erros em $S$ neutralizam o que não deveriam (*beard*, *uterus*).
* Só gênero binário; só uma direção; só inglês. Outras línguas (gênero gramatical!) e outros vieses (raça, etnia, religião: §9) ficam como trabalho futuro — o notebook 10 explora o português.
* O artigo é claro sobre o objetivo mínimo: *machine learning should not be used to inadvertently amplify these biases*.

### Pontos para a apresentação
1. Neutralize = projeção; Equalize = simetrização; Soft = otimização com compromisso. Três operações lineares simples.
2. A **Observação 1** é verificável em uma linha de código — mostre a tabela da seção 2.
3. Utilidade preservada (Tabela 1) é o argumento de venda; "Lipstick on a Pig" é o contraponto obrigatório.

## Referências
- BOLUKBASI, T.; CHANG, K.-W.; ZOU, J.; SALIGRAMA, V.; KALAI, A. **Man is to Computer Programmer as Woman is to Homemaker? Debiasing Word Embeddings.** NIPS, 2016. [arXiv:1607.06520](https://arxiv.org/abs/1607.06520)
- GONEN, H.; GOLDBERG, Y. **Lipstick on a Pig: Debiasing Methods Cover up Systematic Gender Biases in Word Embeddings But do not Remove Them.** NAACL, 2019.